# Data Import

## package import

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as sm

from datetime import datetime, timedelta

## load forecast dataset

In [ ]:
offset = 0

today = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize()

today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

load_forecast = pd.read_csv(f'data/load_frcstd_7_day/load_frcstd_7_day_{today_str}.csv', skiprows=[1])

load_forecast['forecast_load_mw'] = pd.to_numeric(load_forecast['forecast_load_mw'], errors='coerce')

load_forecast =  load_forecast[['evaluated_at_datetime_ept', 'forecast_datetime_ending_ept', 'forecast_area', 'forecast_load_mw']].copy()

load_forecast["forecast_datetime_ending_ept"] = pd.to_datetime(load_forecast["forecast_datetime_ending_ept"])

load_forecast = load_forecast[load_forecast["forecast_datetime_ending_ept"] >= today]


In [4]:
forecast_to_agg = {
    "AE/MIDATL": "AE",
    "AEP": "AEP",
    "AP": "AP",
    "ATSI": "ATSI",
    "BG&E/MIDATL": "BGE",
    "COMED": "COMED",
    "DAYTON": "DAYTON",
    "DEOK": "DEOK",
    "DOMINION": "DOM",
    "DP&L/MIDATL": "DPL",
    "DUQUESNE": "DUQ",
    "EKPC": "EKPC",
    "JCP&L/MIDATL": "JC",
    "METED/MIDATL": "METED",
    "PECO/MIDATL": "PE",
    "PENELEC/MIDATL": "PN",
    "PEPCO/MIDATL": "PEP",
    "PPL/MIDATL": "PL",
    "PSE&G/MIDATL": "PS",
    "RECO/MIDATL": "RE",
    "UGI/MIDATL": "UGI",
    
    # Regions
    "WESTERN_REGION": "WEST_REGION",
    "SOUTHERN_REGION": "SOUTH_REGION",
    "MID_ATLANTIC_REGION": "MIDATL_REGION",

    # Whole system
    "RTO_COMBINED": "RTO"
}

In [5]:
load_forecast["zone"] = load_forecast["forecast_area"].map(forecast_to_agg)

In [6]:
load_forecast = (
    load_forecast
    .sort_values(["forecast_datetime_ending_ept", "zone", "evaluated_at_datetime_ept"])
    .drop_duplicates(subset=["forecast_datetime_ending_ept", "zone"], keep="last")
    .reset_index(drop=True)
)

In [7]:
zone_load_forecast = load_forecast[load_forecast['zone'].isin(['AE', 'AEP', 'AP', 'ATSI', 'BGE', 'COMED', 'DAYTON', 'DEOK', 'DOM', 'DPL', 'DUQ', 'EKPC', 'JC', 'METED', 'PE', 'PN', 'PEP', 'PL', 'PS', 'RE', 'UGI'])].copy()
region_load_forecast = load_forecast[load_forecast['zone'].isin(['WEST_REGION', 'SOUTH_REGION', 'MIDATL_REGION'])].copy()

In [8]:
zone_load_forecast['evaluated_at_datetime_ept'].min(), zone_load_forecast['evaluated_at_datetime_ept'].max()

('2026-05-01 07:17:11.000', '2026-05-01 07:17:11.000')

In [9]:
ugi_mw = (
    zone_load_forecast.loc[zone_load_forecast["zone"] == "UGI", ["forecast_datetime_ending_ept", "forecast_load_mw"]]
    .rename(columns={"forecast_load_mw": "ugi_forecast_load_mw"})
)

zone_load_forecast = zone_load_forecast.merge(ugi_mw, on="forecast_datetime_ending_ept", how="left")

zone_load_forecast.loc[zone_load_forecast["zone"] == "PL", "forecast_load_mw"] = (
    zone_load_forecast.loc[zone_load_forecast["zone"] == "PL", "forecast_load_mw"] +
    zone_load_forecast.loc[zone_load_forecast["zone"] == "PL", "ugi_forecast_load_mw"].fillna(0)
)

zone_load_forecast = (
    zone_load_forecast.loc[zone_load_forecast["zone"] != "UGI"]
    .drop(columns="ugi_forecast_load_mw")
    .reset_index(drop=True)
)

In [10]:
zone_load_forecast.tail()

,evaluated_at_datetime_ept,forecast_datetime_ending_ept,forecast_area,forecast_load_mw,zone
2415,2026-05-01 07:17:11.000,2026-05-08,PEPCO/MIDATL,2564.0,PEP
2416,2026-05-01 07:17:11.000,2026-05-08,PPL/MIDATL,3811.0,PL
2417,2026-05-01 07:17:11.000,2026-05-08,PENELEC/MIDATL,1703.0,PN
2418,2026-05-01 07:17:11.000,2026-05-08,PSE&G/MIDATL,4093.0,PS
2419,2026-05-01 07:17:11.000,2026-05-08,RECO/MIDATL,141.0,RE


In [11]:
mapping = {
    'AE': 'AE',
    'AEP': 'AEP',
    'AP': 'APS',
    'ATSI': 'ATSI',
    'BGE': 'BGE',
    'COMED': 'COMED',
    'DAYTON': 'DAYTON',
    'DEOK': 'DUKE',
    'DOM': 'VEPCO',
    'DPL': 'DPL',
    'DUQ': 'DQE',
    'EKPC': 'EKPC',
    'JC': 'JCPL',
    'METED': 'METED',
    'PE': 'PECO',
    'PEP': 'PEPCO',
    'PL': 'PL',
    'PN': 'PENLC',
    'PS': 'PS',
    'RE': 'RECO'
}

zone_load_forecast['zone'] = zone_load_forecast['zone'].map(mapping)

## weather feature dataset

In [12]:
run_str = (datetime.today() - timedelta(days=offset)).strftime("%y%m%d")  

hist_files = [
    "data/weather/weather_hist_2023.csv",
    "data/weather/weather_hist_2024.csv",
    "data/weather/weather_hist_2025.csv",
    "data/weather/weather_hist_2026.csv",
]

forecast_file = f"data/weather/weather_forecast_{run_str}.csv"

# --- load historical ---
hist_weather = pd.concat([pd.read_csv(f) for f in hist_files], ignore_index=True)
hist_weather.drop(columns=["weather_code"], errors="ignore", inplace=True)
hist_weather["time"] = pd.to_datetime(hist_weather["time"])

# keep only hist data up to run_str
run_dt = pd.to_datetime(run_str, format="%y%m%d")
hist_weather = hist_weather[hist_weather["time"] <= run_dt].copy()

# --- load forecast ---
forecast_weather = pd.read_csv(forecast_file)
forecast_weather.drop(columns=["weather_code"], errors="ignore", inplace=True)
forecast_weather["time"] = pd.to_datetime(forecast_weather["time"])

# keep only forecast data from forecast_run_str
forecast_weather = forecast_weather[
    forecast_weather["time"] > run_dt
].copy()

# dedupe
dedupe_cols = ["zone", "weather_station", "lat", "lon", "time"]
hist_weather = hist_weather.drop_duplicates(subset=dedupe_cols, keep="first").reset_index(drop=True)
forecast_weather = forecast_weather.drop_duplicates(subset=dedupe_cols, keep="first").reset_index(drop=True)

# --- fill forecast nulls with hour average ---
forecast_weather["hour"] = forecast_weather["time"].dt.hour

weather_vars = [
    'temperature_2m', 
    'apparent_temperature', 
    'dew_point_2m',
    'relative_humidity_2m', 
    'precipitation',
    'rain', 
    'snowfall', 
    'snow_depth', 
    'cloud_cover', 
    'cloud_cover_low',
    'cloud_cover_mid', 
    'cloud_cover_high', 
    'wind_speed_10m',
    'wind_direction_10m', 
    'wind_gusts_10m', 
    'surface_pressure', 
    'pressure_msl',
    'et0_fao_evapotranspiration',
    'vapour_pressure_deficit',
    'shortwave_radiation', 
    'direct_radiation', 
    'diffuse_radiation',
    'global_tilted_irradiance', 
    'direct_normal_irradiance',
    'terrestrial_radiation'
]

# hour average within each zone/station/hour
hourly_avg = (
    forecast_weather.groupby(["zone", "weather_station", "hour"])[weather_vars]
    .mean()
    .reset_index()
)

forecast_weather = forecast_weather.merge(
    hourly_avg,
    on=["zone", "weather_station", "hour"],
    how="left",
    suffixes=("", "_hour_avg")
)

for col in weather_vars:
    forecast_weather[col] = forecast_weather[col].fillna(forecast_weather[f"{col}_hour_avg"])

# drop helper columns
drop_cols = (
    ["hour"]
    + [f"{c}_hour_avg" for c in weather_vars]
)
forecast_weather.drop(columns=drop_cols, inplace=True, errors="ignore")

# --- final zonal_weather ---
zonal_weather = pd.concat([hist_weather, forecast_weather], ignore_index=True)
zonal_weather = zonal_weather.sort_values(
    ["zone", "weather_station", "time"]
).reset_index(drop=True)

In [13]:
for var in weather_vars:
    zonal_weather[f"{var}_weighted"] = (
        zonal_weather[var] * zonal_weather["weather_weight"]
    )

zonal_weather = (
    zonal_weather.groupby(["time", "zone"])[
        [f"{v}_weighted" for v in weather_vars]
    ]
    .sum()
    .reset_index()
)

zonal_weather = zonal_weather.rename(columns={
    "temperature_2m_weighted": "temperature_2m",
    "apparent_temperature_weighted": "apparent_temperature",
    "dew_point_2m_weighted": "dew_point_2m",
    "relative_humidity_2m_weighted": "relative_humidity_2m",
    "precipitation_weighted": "precipitation",
    "rain_weighted": "rain",
    "snowfall_weighted": "snowfall",
    "snow_depth_weighted": "snow_depth",
    "cloud_cover_weighted": "cloud_cover",
    "cloud_cover_low_weighted": "cloud_cover_low",
    "cloud_cover_mid_weighted": "cloud_cover_mid",
    "cloud_cover_high_weighted": "cloud_cover_high",
    "wind_speed_10m_weighted": "wind_speed_10m",
    "wind_direction_10m_weighted": "wind_direction_10m",
    "wind_gusts_10m_weighted": "wind_gusts_10m",
    "surface_pressure_weighted": "surface_pressure",
    "pressure_msl_weighted": "pressure_msl",
    "et0_fao_evapotranspiration_weighted": "et0_fao_evapotranspiration",
    "vapour_pressure_deficit_weighted": "vapour_pressure_deficit",
    'shortwave_radiation_weighted': 'shortwave_radiation',
    'direct_radiation_weighted': 'direct_radiation',
    'diffuse_radiation_weighted': 'diffuse_radiation',
    'global_tilted_irradiance_weighted': 'global_tilted_irradiance',
    'direct_normal_irradiance_weighted': 'direct_normal_irradiance',
    'terrestrial_radiation_weighted': 'terrestrial_radiation'
})

zonal_weather.rename(columns={"time": "datetime_ending_ept"}, inplace=True)



In [14]:
zonal_weather.tail()

,datetime_ending_ept,zone,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,snow_depth,...,surface_pressure,pressure_msl,et0_fao_evapotranspiration,vapour_pressure_deficit,shortwave_radiation,direct_radiation,diffuse_radiation,global_tilted_irradiance,direct_normal_irradiance,terrestrial_radiation
616912,2026-05-09,PL,5.9500,3.3750,0.9750,70.750,0.0,0.0,0.0,0.0,...,991.9750,1011.5750,0.00250,0.27750,0.0,0.0,0.0,0.0,0.0,0.0
616913,2026-05-09,PS,11.9000,9.1000,2.6000,53.000,0.0,0.0,0.0,0.0,...,1010.9000,1011.1000,0.03000,0.66000,0.0,0.0,0.0,0.0,0.0,0.0
616914,2026-05-09,RECO,11.9000,9.1000,2.6000,53.000,0.0,0.0,0.0,0.0,...,1010.9000,1011.1000,0.03000,0.66000,0.0,0.0,0.0,0.0,0.0,0.0
616915,2026-05-09,UGI,3.8000,0.7000,-0.1000,76.000,0.0,0.0,0.0,0.0,...,977.0000,1011.5000,0.00000,0.19000,0.0,0.0,0.0,0.0,0.0,0.0
616916,2026-05-09,VEPCO,12.1212,9.7902,5.2614,62.937,0.0,0.0,0.0,0.0,...,1006.8921,1012.4865,0.02331,0.52947,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
zone_map = {
    "AE": "AE",
    "AEP": "AEP",
    "APS": "AP",
    "ATSI": "ATSI",
    "BGE": "BGE",
    "COMED": "COMED",
    "DAYTON": "DAYTON",
    "DPL": "DPL",
    "DQE": "DUQ",
    "EKPC": "EKPC",
    "JCPL": "JC",
    "METED": "METED",
    "PECO": "PE",
    "PENLC": "PN",
    "PEPCO": "PEP",
    "PL": "PL",
    "PS": "PS",
    "RECO": "RE",
    "UGI": "UGI",
    "VEPCO": "DOM",
    "DUKE": "DEOK"
}

zonal_weather["agg_NodeName"] = zonal_weather["zone"].map(zone_map)

## calendar feature dataset

In [16]:
import holidays

zonal_weather['datetime_ending_ept'] = pd.to_datetime(zonal_weather['datetime_ending_ept'])

zonal_weather['date'] = zonal_weather['datetime_ending_ept'].dt.date
zonal_weather['year'] = zonal_weather['datetime_ending_ept'].dt.year
zonal_weather['month'] = zonal_weather['datetime_ending_ept'].dt.month
zonal_weather['day'] = zonal_weather['datetime_ending_ept'].dt.day
zonal_weather['hour'] = zonal_weather['datetime_ending_ept'].dt.hour
zonal_weather['day_of_week'] = zonal_weather['datetime_ending_ept'].dt.dayofweek

zonal_weather['is_weekend'] = zonal_weather['day_of_week'].isin([5,6]).astype(int)

us_holidays = holidays.US(observed=True)

zonal_weather['is_holiday'] = zonal_weather['datetime_ending_ept'].apply(
    lambda x: int(x.date() in us_holidays)
)

zonal_weather['WkDayBeforeHol'] = zonal_weather['datetime_ending_ept'].apply(
    lambda x: int((x + pd.Timedelta(days=1)) in us_holidays and pd.Timestamp(x).dayofweek < 5)
)

zonal_weather['WkDayAfterHol'] = zonal_weather['datetime_ending_ept'].apply(
    lambda x: int((x - pd.Timedelta(days=1)) in us_holidays and pd.Timestamp(x).dayofweek < 5)
)


from dateutil.easter import easter

def build_event_calendar(start_year, end_year):
    rows = []

    # Federal holidays
    us = holidays.US(years=range(start_year, end_year + 1), observed=True)
    for d, name in us.items():
        rows.append((pd.Timestamp(d), name))

    for y in range(start_year, end_year + 1):
        # Easter
        e = pd.Timestamp(easter(y))
        rows += [
            (e - pd.Timedelta(days=2), "Good Friday"),
            (e, "Easter"),
        ]

        # Fixed-date events 
        rows += [
            (pd.Timestamp(f"{y}-12-24"), "Christmas Eve"),
            (pd.Timestamp(f"{y}-12-31"), "New Year's Eve"),
            (pd.Timestamp(f"{y}-10-31"), "Halloween")
        ]

        # Thanksgiving-based 
        tg = [d for d, n in holidays.US(years=[y]).items() if "Thanksgiving" in n][0]

        rows += [
            (pd.Timestamp(tg) + pd.Timedelta(days=1), "Black Friday"),
            (pd.Timestamp(tg) + pd.Timedelta(days=4), "Cyber Monday"),
        ]

    event_df = pd.DataFrame(rows, columns=["date", "event_name"])

    # combine duplicates
    event_df = (
        event_df.groupby("date")["event_name"]
        .apply(lambda x: ", ".join(sorted(set(x))))
        .reset_index()
        .sort_values("date")
    )

    return event_df

# build calendar
start_year = zonal_weather["datetime_ending_ept"].dt.year.min()
end_year   = zonal_weather["datetime_ending_ept"].dt.year.max()

event_df = build_event_calendar(start_year, end_year)

# prepare join key
zonal_weather["date"] = pd.to_datetime(zonal_weather["datetime_ending_ept"]).dt.normalize()

# merge
zonal_weather = zonal_weather.merge(event_df, on="date", how="left")

# flag
zonal_weather["is_event"] = zonal_weather["event_name"].notna().astype(int)

zonal_weather.drop(columns=["event_name"], inplace=True)



In [17]:
zonal_wc = zonal_weather.copy()

zonal_wc = zonal_wc[zonal_wc['agg_NodeName'] != 'UGI'].copy()

In [18]:
zonal_wc.tail()

,datetime_ending_ept,zone,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,snow_depth,...,year,month,day,hour,day_of_week,is_weekend,is_holiday,WkDayBeforeHol,WkDayAfterHol,is_event
616911,2026-05-09,PEPCO,12.5000,10.4000,4.4000,58.000,0.0,0.0,0.0,0.0,...,2026,5,9,0,5,1,0,0,0,0
616912,2026-05-09,PL,5.9500,3.3750,0.9750,70.750,0.0,0.0,0.0,0.0,...,2026,5,9,0,5,1,0,0,0,0
616913,2026-05-09,PS,11.9000,9.1000,2.6000,53.000,0.0,0.0,0.0,0.0,...,2026,5,9,0,5,1,0,0,0,0
616914,2026-05-09,RECO,11.9000,9.1000,2.6000,53.000,0.0,0.0,0.0,0.0,...,2026,5,9,0,5,1,0,0,0,0
616916,2026-05-09,VEPCO,12.1212,9.7902,5.2614,62.937,0.0,0.0,0.0,0.0,...,2026,5,9,0,5,1,0,0,0,0


## load historical dataset

In [2]:
hourly_load = pd.read_csv(f'data/raw_pjm_hrl_load_metered/raw_pjm_hrl_load_metered_{today_str}.csv', skiprows=[1], usecols=lambda c: c not in ["datetime_beginning_utc", "auto_key", "is_verified", "insert_datetime"])

hourly_load["MW"] = pd.to_numeric(hourly_load["MW"], errors='coerce')

NameError: name 'today_str' is not defined

In [20]:
hourly_load["datetime_beginning_ept"] = pd.to_datetime(hourly_load["datetime_beginning_ept"])
hourly_load["datetime_beginning_ept"] += pd.Timedelta(hours=1)

hourly_load = hourly_load.rename(columns={
    "datetime_beginning_ept": "datetime_ending_ept"
})

# desired first columns
first_cols = ["datetime_ending_ept", "zone"]

# reorder dataframe
hourly_load = hourly_load[first_cols + [c for c in hourly_load.columns if c not in first_cols]]


In [21]:
hourly_zone_load = hourly_load[hourly_load['zone'].isin(['AE', 'AEP', 'AP', 'ATSI', 'BC', 'CE', 'DAY', 'DEOK', 'DOM', 'DPL', 'DUQ', 'EKPC', 'JC', 'ME', 'OVEC', 'PE', 'PN', 'PEP', 'PL', 'PS', 'RECO'])].copy()

In [22]:
hourly_zone_load.drop_duplicates(subset=['datetime_ending_ept', 'zone', 'nerc_region', 'mkt_region', 'load_area'], keep='first', inplace=True)

hourly_zone_load = (
    hourly_zone_load
    .groupby(['datetime_ending_ept', 'zone'], as_index=False)
    .agg({
        'MW': 'sum',
        'nerc_region': 'first',
        'mkt_region': 'first',
    })
)


In [23]:
last_time_per_zone = (
    hourly_zone_load
    .groupby("zone")["datetime_ending_ept"]
    .max()
    .reset_index()
)

In [24]:
last_time_per_zone 

,zone,datetime_ending_ept
0,AE,2026-05-01
1,AEP,2026-05-01
2,AP,2026-05-01
3,ATSI,2026-05-01
4,BC,2026-05-01
5,CE,2026-05-01
6,DAY,2026-05-01
7,DEOK,2026-05-01
8,DOM,2026-05-01
9,DPL,2026-05-01


In [25]:
hourly_zone_load.tail()

,datetime_ending_ept,zone,MW,nerc_region,mkt_region
612775,2026-05-01,PEP,2406.527110,RFC,MIDATL
612776,2026-05-01,PL,3872.448902,RFC,MIDATL
612777,2026-05-01,PN,1691.624000,RFC,MIDATL
612778,2026-05-01,PS,4056.198000,RFC,MIDATL
612779,2026-05-01,RECO,131.509000,RFC,MIDATL


In [26]:
append_load = pd.read_csv(f"data/pjm_All_Instantaneous_Load_rt5/pjm_All_Instantaneous_Load_rt5_{today_str}.csv", skiprows=[1])

append_load["instantaneous_load"] = pd.to_numeric(append_load["instantaneous_load"], errors='coerce')

append_load["datetime_beginning_ept"] = pd.to_datetime(append_load["datetime_beginning_ept"])

append_load = append_load[append_load['area'] != 'UG'].copy()

In [27]:
append_load = (
    append_load
    .assign(
        hour_beginning_ept=lambda x: x["datetime_beginning_ept"].dt.floor("h"),
        datetime_ending_ept=lambda x: x["datetime_beginning_ept"].dt.floor("h") + pd.Timedelta(hours=1)
    )
    .groupby(["area", "datetime_ending_ept"], as_index=False)["instantaneous_load"]
    .mean()
    .rename(columns={"instantaneous_load": "load_mw_hourly_avg"})
)

In [28]:
append_load.tail()

,area,datetime_ending_ept,load_mw_hourly_avg
71035,RECO,2026-05-04 05:00:00,120.854684
71036,RECO,2026-05-04 06:00:00,131.534169
71037,RECO,2026-05-04 07:00:00,136.342021
71038,RECO,2026-05-04 08:00:00,145.700077
71039,RECO,2026-05-04 09:00:00,147.101658


In [29]:
zone_mapping = {
    'AE': 'AE',
    'AEP': 'AEP',
    'APS': 'AP',
    'ATSI': 'ATSI',
    'BC': 'BC',
    'COMED': 'CE',
    'DAYTON': 'DAY',
    'DEOK': 'DEOK',
    'DOM': 'DOM',
    'DPL': 'DPL',
    'DUQ': 'DUQ',
    'EKPC': 'EKPC',
    'JC': 'JC',
    'ME': 'ME',
    'PE': 'PE',
    'PEP': 'PEP',
    'PL': 'PL',
    'PN': 'PN',
    'PS': 'PS',
    'RECO': 'RECO'
}

append_load["zone"] = append_load["area"].map(zone_mapping)

append_load.dropna(subset=["zone"], inplace=True)

In [30]:
# find the latest timestamp already in hourly_zone_load
max_dt_per_zone = (
    hourly_zone_load
    .groupby("zone")["datetime_ending_ept"]
    .max()
)

max_dt_hourly = max_dt_per_zone.min()

# keep only rows from append_load that are after that timestamp
append_load_new = append_load.loc[
    append_load["datetime_ending_ept"] > max_dt_hourly,
    ["datetime_ending_ept", "zone", "load_mw_hourly_avg"]
].copy()

# rename to match hourly_zone_load schema
append_load_new = append_load_new.rename(columns={"load_mw_hourly_avg": "MW"})
append_load_new["nerc_region"] = append_load_new["zone"]
append_load_new["mkt_region"] = append_load_new["zone"]

# append to hourly_zone_load
hourly_zone_load = pd.concat(
    [hourly_zone_load, append_load_new[["datetime_ending_ept", "zone", "MW", "nerc_region", "mkt_region"]]],
    ignore_index=True
).sort_values("datetime_ending_ept").reset_index(drop=True)

In [31]:
# get OVEC MW by timestamp
ovec_mw = (
    hourly_zone_load.loc[hourly_zone_load["zone"] == "OVEC", ["datetime_ending_ept", "MW"]]
    .rename(columns={"MW": "OVEC_MW"})
)

# attach OVEC MW to matching rows
hourly_zone_load = hourly_zone_load.merge(ovec_mw, on="datetime_ending_ept", how="left")

# add OVEC MW only to AEP rows
hourly_zone_load.loc[hourly_zone_load["zone"] == "AEP", "MW"] = (
    hourly_zone_load.loc[hourly_zone_load["zone"] == "AEP", "MW"] +
    hourly_zone_load.loc[hourly_zone_load["zone"] == "AEP", "OVEC_MW"].fillna(0)
)

# remove OVEC rows and helper column
hourly_zone_load = (
    hourly_zone_load.loc[hourly_zone_load["zone"] != "OVEC"]
    .drop(columns="OVEC_MW")
    .reset_index(drop=True)
)

In [32]:
hourly_zone_load.head()

,datetime_ending_ept,zone,MW,nerc_region,mkt_region
0,2023-01-01 01:00:00,AE,936.655989,RFC,MIDATL
1,2023-01-01 01:00:00,RECO,128.998000,RFC,MIDATL
2,2023-01-01 01:00:00,PS,3983.133100,RFC,MIDATL
3,2023-01-01 01:00:00,PN,1574.133100,RFC,MIDATL
4,2023-01-01 01:00:00,PL,3754.025000,RFC,MIDATL


In [32]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# plot_df = (
#     hourly_zone_load.groupby(["datetime_ending_ept", "zone"], as_index=False)["MW"]
#     .sum()
#     .sort_values(["zone", "datetime_ending_ept"])
# )

# zones = sorted(plot_df["zone"].dropna().unique())

# nrows, ncols = 5, 4
# max_plots = nrows * ncols

# # keep at most 20 zones for a 5x4 layout
# zones_to_plot = zones[:max_plots]

# fig = make_subplots(
#     rows=nrows,
#     cols=ncols,
#     subplot_titles=zones_to_plot,
#     shared_xaxes=False,
#     shared_yaxes=False,
#     vertical_spacing=0.06,
#     horizontal_spacing=0.05
# )

# for i, z in enumerate(zones_to_plot):
#     r = i // ncols + 1
#     c = i % ncols + 1
    
#     temp = plot_df[plot_df["zone"] == z]
    
#     fig.add_trace(
#         go.Scatter(
#             x=temp["datetime_ending_ept"],
#             y=temp["MW"],
#             mode="lines",
#             name=z,
#             showlegend=False
#         ),
#         row=r,
#         col=c
#     )

# fig.update_layout(
#     height=1500,
#     width=2000,
#     title="MW over Time by Zone",
#     template="plotly_white"
# )

# fig.update_xaxes(title_text="Time")
# fig.update_yaxes(title_text="MW")

# fig.show()

In [33]:
df = hourly_zone_load.copy()
df["datetime_ending_ept"] = pd.to_datetime(df["datetime_ending_ept"])

df["year"] = df["datetime_ending_ept"].dt.year
df["month"] = df["datetime_ending_ept"].dt.month

# March only
march = df[df["month"] == 3]

# Median MW per (zone, year)
march_median = (
    march.groupby(["zone", "year"])["MW"]
    .median()
    .sort_index()
)

# YoY growth per zone
march_growth = march_median.groupby(level=0).pct_change()

# Convert to dict: {(zone, year): growth}
growth_dict = march_growth.dropna().to_dict()

df["date"] = df["datetime_ending_ept"].dt.floor("D")
t0 = df["date"].max()

def growth_factor(dt, zone, t0, growth_dict):
    if dt >= t0:
        return 1.0

    factor = 1.0
    current = dt

    while current < t0:
        year_end = pd.Timestamp(year=current.year + 1, month=1, day=1)
        segment_end = min(year_end, t0)

        frac_year = (segment_end - current).days / 365.25

        # Key: (zone, next_year)
        r = growth_dict.get((zone, current.year + 1), 0.0)

        factor *= (1 + r) ** frac_year
        current = segment_end

    return factor

df["growth_factor"] = df.apply(
    lambda row: growth_factor(row["date"], row["zone"], t0, growth_dict),
    axis=1
)

df["MW_normalized"] = df["MW"] * df["growth_factor"]

hourly_zone_load = df[
    ["datetime_ending_ept", "zone", "MW_normalized", "nerc_region", "mkt_region"]
].rename(columns={"MW_normalized": "MW"})

In [35]:
today_hour = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize() + pd.Timedelta(hours=9)

hourly_zone_load = hourly_zone_load[hourly_zone_load['datetime_ending_ept'] <= today_hour].copy()

hourly_zone_load.tail()

,datetime_ending_ept,zone,MW,nerc_region,mkt_region
584735,2026-05-03 09:00:00,DPL,1515.714169,DPL,DPL
584736,2026-05-03 09:00:00,DUQ,1268.414554,DUQ,DUQ
584737,2026-05-03 09:00:00,DOM,13332.301462,DOM,DOM
584738,2026-05-03 09:00:00,JC,1653.966038,JC,JC
584739,2026-05-03 09:00:00,ATSI,6595.867185,ATSI,ATSI


In [36]:
growth_dict

{('AE', 2024): -0.02320229240578886,
 ('AE', 2025): -0.024385972194975958,
 ('AE', 2026): 0.046650611876898784,
 ('AEP', 2024): -0.01662150123908479,
 ('AEP', 2025): 0.038715076419566286,
 ('AEP', 2026): 0.08741909844528317,
 ('AP', 2024): -0.04696184710998452,
 ('AP', 2025): 0.00209601713526264,
 ('AP', 2026): 0.014348068831991245,
 ('ATSI', 2024): -0.04560220555820227,
 ('ATSI', 2025): 0.019351130424708263,
 ('ATSI', 2026): -0.0025908586940730505,
 ('BC', 2024): -0.03706938451206021,
 ('BC', 2025): -0.015962024073330472,
 ('BC', 2026): 0.03461538334856673,
 ('CE', 2024): -0.05693391378641932,
 ('CE', 2025): 0.019202595379379517,
 ('CE', 2026): 0.012138086796473546,
 ('DAY', 2024): -0.04395783446121815,
 ('DAY', 2025): 0.020714303993831784,
 ('DAY', 2026): 0.009554571641950282,
 ('DEOK', 2024): -0.03081419427181742,
 ('DEOK', 2025): 0.018331206610240347,
 ('DEOK', 2026): -0.028536266273621047,
 ('DOM', 2024): 0.028148924708367495,
 ('DOM', 2025): 0.06688769817471352,
 ('DOM', 2026): 0

In [37]:
zone_mapping = {
    "AE": "AE",
    "AEP": "AEP",
    "AP": "APS",
    "ATSI": "ATSI",
    "BC": "BGE",
    "CE": "COMED",
    "DAY": "DAYTON",
    "DEOK": "DUKE", 
    "DOM": "VEPCO",
    "DPL": "DPL",
    "DUQ": "DQE",
    "EKPC": "EKPC",
    "JC": "JCPL",
    "ME": "METED",
    "PE": "PECO",
    "PEP": "PEPCO",
    "PL": "PL",
    "PN": "PENLC",
    "PS": "PS",
    "RECO": "RECO"
}

hourly_zone_load["zone"] = hourly_zone_load["zone"].map(zone_mapping)

In [38]:
# keep only the columns you want to bring into zonal_wc
mw_region = hourly_zone_load[["datetime_ending_ept", "zone", "MW"]].copy()

# merge into zonal_wc
zonal_wcl = zonal_wc.merge(
    mw_region,
    on=["datetime_ending_ept", "zone"],
    how="left"
)


In [39]:
zonal_wcl.tail()

,datetime_ending_ept,zone,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,snow_depth,...,month,day,hour,day_of_week,is_weekend,is_holiday,WkDayBeforeHol,WkDayAfterHol,is_event,MW
587535,2026-05-09,PEPCO,12.5000,10.4000,4.4000,58.000,0.0,0.0,0.0,0.0,...,5,9,0,5,1,0,0,0,0,NaN
587536,2026-05-09,PL,5.9500,3.3750,0.9750,70.750,0.0,0.0,0.0,0.0,...,5,9,0,5,1,0,0,0,0,NaN
587537,2026-05-09,PS,11.9000,9.1000,2.6000,53.000,0.0,0.0,0.0,0.0,...,5,9,0,5,1,0,0,0,0,NaN
587538,2026-05-09,RECO,11.9000,9.1000,2.6000,53.000,0.0,0.0,0.0,0.0,...,5,9,0,5,1,0,0,0,0,NaN
587539,2026-05-09,VEPCO,12.1212,9.7902,5.2614,62.937,0.0,0.0,0.0,0.0,...,5,9,0,5,1,0,0,0,0,NaN


## Zone cutoff

In [40]:
# # DOM BGE AEP PN
# zonal_wcl = zonal_wcl[zonal_wcl['zone'].isin(['VEPCO', 'BGE', 'AEP', 'PENLC'])].copy()


In [41]:
# zonal_wcl.tail()

# Vanila Model

## Alert signal

In [42]:
# alert
alert = pd.read_csv("data/emergencymessages/alerts.csv")
alert["start"] = pd.to_datetime(alert["start"].astype(str).str[:-6])
alert["end"]   = pd.to_datetime(alert["end"].astype(str).str[:-6])

In [43]:
zonal_wcl["is_alert"] = 0

for _, row in alert.iterrows():
    mask = (
        (zonal_wcl["datetime_ending_ept"] >= row["start"]) &
        (zonal_wcl["datetime_ending_ept"] <= row["end"])
    )
    zonal_wcl.loc[mask, "is_alert"] = 1

# zonal_wcl = zonal_wcl[zonal_wcl['is_alert'] == 0].copy()

In [44]:
warning = pd.read_csv("data/emergencymessages/warnings.csv")
warning["start"] = pd.to_datetime(warning["start"].astype(str).str[:-6])
warning["end"]   = pd.to_datetime(warning["end"].astype(str).str[:-6])

In [45]:
zonal_wcl["is_warning"] = 0

for _, row in warning.iterrows():
    mask = (
        (zonal_wcl["datetime_ending_ept"] >= row["start"]) &
        (zonal_wcl["datetime_ending_ept"] <= row["end"])
    )
    zonal_wcl.loc[mask, "is_warning"] = 1

# zonal_wcl = zonal_wcl[zonal_wcl['is_warning'] == 0].copy()

In [46]:
action = pd.read_csv("data/emergencymessages/actions.csv")
action["start"] = pd.to_datetime(action["start"].astype(str).str[:-6])
action["end"]   = pd.to_datetime(action["end"].astype(str).str[:-6])

In [47]:
zonal_wcl["is_action"] = 0

for _, row in action.iterrows():
    mask = (
        (zonal_wcl["datetime_ending_ept"] >= row["start"]) &
        (zonal_wcl["datetime_ending_ept"] <= row["end"])
    )
    zonal_wcl.loc[mask, "is_action"] = 1

In [48]:
zonal_wcl.tail()

,datetime_ending_ept,zone,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,snow_depth,...,day_of_week,is_weekend,is_holiday,WkDayBeforeHol,WkDayAfterHol,is_event,MW,is_alert,is_warning,is_action
587535,2026-05-09,PEPCO,12.5000,10.4000,4.4000,58.000,0.0,0.0,0.0,0.0,...,5,1,0,0,0,0,NaN,0,0,0
587536,2026-05-09,PL,5.9500,3.3750,0.9750,70.750,0.0,0.0,0.0,0.0,...,5,1,0,0,0,0,NaN,0,0,0
587537,2026-05-09,PS,11.9000,9.1000,2.6000,53.000,0.0,0.0,0.0,0.0,...,5,1,0,0,0,0,NaN,0,0,0
587538,2026-05-09,RECO,11.9000,9.1000,2.6000,53.000,0.0,0.0,0.0,0.0,...,5,1,0,0,0,0,NaN,0,0,0
587539,2026-05-09,VEPCO,12.1212,9.7902,5.2614,62.937,0.0,0.0,0.0,0.0,...,5,1,0,0,0,0,NaN,0,0,0


## Temp Feature Construction  

In [49]:
daily_temp = (
    zonal_wcl.groupby("date", as_index=False)["temperature_2m"]
    .agg(temp_min="min", temp_max="max")
)

zonal_wcl = zonal_wcl.merge(daily_temp, on="date", how="left")

zonal_wcl["temp_f"] = zonal_wcl["temperature_2m"] * 9/5 + 32

base_temp = 65

zonal_wcl["HDD"] = np.maximum(base_temp - zonal_wcl["temp_f"], 0)
zonal_wcl["CDD"] = np.maximum(zonal_wcl["temp_f"] - base_temp, 0)

zonal_wcl["HDD_wind"] = zonal_wcl["HDD"] * zonal_wcl["wind_speed_10m"]
zonal_wcl["CDD_cloud"] = zonal_wcl["CDD"] * (1 - zonal_wcl["cloud_cover"] / 100)

zonal_wcl["wind_dir_10m_sin"] = np.sin(np.deg2rad(zonal_wcl["wind_direction_10m"]))
zonal_wcl["wind_dir_10m_cos"] = np.cos(np.deg2rad(zonal_wcl["wind_direction_10m"]))

zonal_wcl["wind_chill"] = (
    35.74
    + 0.6215 * zonal_wcl["temp_f"]
    - 35.75 * (zonal_wcl["wind_speed_10m"] ** 0.16)
    + 0.4275 * zonal_wcl["temp_f"] * (zonal_wcl["wind_speed_10m"] ** 0.16)
)

RH = zonal_wcl["relative_humidity_2m"]
T = zonal_wcl["temp_f"]

zonal_wcl["heat_index_f"] = (
    -42.379
    + 2.04901523 * T
    + 10.14333127 * RH
    - 0.22475541 * T * RH
    - 0.00683783 * T**2
    - 0.05481717 * RH**2
    + 0.00122874 * T**2 * RH
    + 0.00085282 * T * RH**2
    - 0.00000199 * T**2 * RH**2
)

zonal_wcl["feels_like_temp"] = np.where(
    (zonal_wcl["temp_f"] <= 50) & (zonal_wcl["wind_speed_10m"] > 3),
    zonal_wcl["wind_chill"],
    np.where(
        (zonal_wcl["temp_f"] >= 80) & (RH >= 40),
        zonal_wcl["heat_index_f"],
        zonal_wcl["temp_f"]
    )
)

# hour-to-hour absolute temperature change
zonal_wcl["temp_diff_1h_abs"] = zonal_wcl["temperature_2m"].diff().abs()

# cumulative total variation over past 6 hours
zonal_wcl["temp_total_variation_6h"] = (
    zonal_wcl["temp_diff_1h_abs"].rolling(6).sum()
)

# cumulative total variation over past 12 hours
zonal_wcl["temp_total_variation_12h"] = (
    zonal_wcl["temp_diff_1h_abs"].rolling(12).sum()
)

# cumulative total variation over past 24 hours
zonal_wcl["temp_total_variation_24h"] = (
    zonal_wcl["temp_diff_1h_abs"].rolling(24).sum()
)

zonal_wcl["temp_fcst_1h"] = zonal_wcl["temperature_2m"].shift(-1)

horizon = 6

zonal_wcl["temp_lead_mean_6h"] = sum(
    zonal_wcl["temperature_2m"].shift(-i)
    for i in range(1, horizon + 1)
) / horizon

zonal_wcl.drop(columns=["wind_chill", "temp_diff_1h_abs", "heat_index_f"], inplace=True)


In [50]:
zonal_wcl[zonal_wcl['zone'] == "RECO"].tail(170)

,datetime_ending_ept,zone,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,snow_depth,...,HDD_wind,CDD_cloud,wind_dir_10m_sin,wind_dir_10m_cos,feels_like_temp,temp_total_variation_6h,temp_total_variation_12h,temp_total_variation_24h,temp_fcst_1h,temp_lead_mean_6h
584158,2026-05-01 23:00:00,RECO,12.6,10.1,5.8,63.0,0.0,0.0,0.0,0.0,...,97.008,0.0,-0.207912,-9.781476e-01,54.680000,23.40,44.180,95.1042,12.4875,8.025133
584178,2026-05-02 00:00:00,RECO,11.7,9.0,5.2,64.0,0.0,0.0,0.0,0.0,...,120.594,0.0,-0.939693,-3.420201e-01,53.060000,19.90,41.530,87.9434,11.9880,7.540200
584198,2026-05-02 01:00:00,RECO,10.9,8.4,5.5,70.0,0.1,0.1,0.0,0.0,...,119.082,0.0,-0.891007,-4.539905e-01,51.620000,19.80,42.560,84.1240,11.5551,7.453333
584218,2026-05-02 02:00:00,RECO,10.2,8.1,7.1,81.0,0.0,0.0,0.0,0.0,...,140.544,0.0,-0.766044,-6.427876e-01,50.360000,17.75,41.340,82.7004,11.5218,7.456433
584238,2026-05-02 03:00:00,RECO,10.0,8.1,7.3,83.0,0.0,0.0,0.0,0.0,...,127.500,0.0,-0.615661,-7.880108e-01,46.570135,16.85,40.080,83.3500,11.2221,6.953067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587458,2026-05-08 20:00:00,RECO,15.1,10.4,2.6,43.0,0.0,0.0,0.0,0.0,...,119.892,0.0,-0.945519,-3.255682e-01,59.180000,23.80,32.626,63.5930,14.6520,12.060633
587478,2026-05-08 21:00:00,RECO,14.2,10.1,3.0,47.0,0.0,0.0,0.0,0.0,...,125.736,0.0,-0.961262,-2.756374e-01,57.560000,25.20,32.932,65.1636,13.6863,11.337883
587498,2026-05-08 22:00:00,RECO,13.4,9.9,3.1,50.0,0.0,0.0,0.0,0.0,...,111.888,0.0,-1.000000,-1.836970e-16,56.120000,25.85,33.020,65.4620,12.9870,10.808350
587518,2026-05-08 23:00:00,RECO,12.6,9.5,3.0,52.0,0.0,0.0,0.0,0.0,...,101.136,0.0,-0.956305,2.923717e-01,54.680000,26.05,33.410,65.4822,12.4542,10.385050


## Lag Feature Construction

In [51]:

zonal_wcl = zonal_wcl.sort_values(["zone", "datetime_ending_ept"]).copy()
zonal_wcl["datetime_ending_ept"] = pd.to_datetime(zonal_wcl["datetime_ending_ept"])

def add_zone_features(df):
    df = df.sort_values("datetime_ending_ept").copy()

    # hour of target timestamp
    df["hour"] = df["datetime_ending_ept"].dt.hour

    # set day_shift
    df["day_shift"] = np.where(df["hour"].isin(range(1, 10)), 1, 2)

    # lookup index from historical MW for this zone only
    hist = (
        df[["datetime_ending_ept", "MW"]]
        .drop_duplicates("datetime_ending_ept")
        .sort_values("datetime_ending_ept")
        .set_index("datetime_ending_ept")
    )

    # choose lag hours to create
    lag_hours = [1, 2, 3, 6, 12, 24, 36, 48, 72, 96, 120, 144, 168]

    # build lag lookup timestamps + features
    helper_cols = []
    feature_cols = []

    for lag in lag_hours:
        time_col = f"lag{lag}_time"
        feat_col = f"MW_lag_{lag}"

        df[time_col] = (
            df["datetime_ending_ept"]
            - pd.to_timedelta(df["day_shift"], unit="D")
            - pd.Timedelta(hours=lag)
        )

        df[feat_col] = hist["MW"].reindex(df[time_col]).to_numpy()

        helper_cols.append(time_col)
        feature_cols.append(feat_col)

    # rolling stats based only on historical series for this zone
    hist["MW_roll_24_base"] = hist["MW"].shift(1).rolling(24).mean()
    hist["MW_roll_48_base"] = hist["MW"].shift(1).rolling(48).mean()
    hist["MW_roll_72_base"] = hist["MW"].shift(1).rolling(72).mean()
    hist["MW_roll_168_base"] = hist["MW"].shift(1).rolling(168).mean()

    hist["MW_std_24_base"] = hist["MW"].shift(1).rolling(24).std()
    hist["MW_std_168_base"] = hist["MW"].shift(1).rolling(168).std()

    # rolling lookup time
    df["roll_time"] = (
        df["datetime_ending_ept"]
        - pd.to_timedelta(df["day_shift"], unit="D")
    )

    df["MW_roll_24"] = hist["MW_roll_24_base"].reindex(df["roll_time"]).to_numpy()
    df["MW_roll_48"] = hist["MW_roll_48_base"].reindex(df["roll_time"]).to_numpy()
    df["MW_roll_72"] = hist["MW_roll_72_base"].reindex(df["roll_time"]).to_numpy()
    df["MW_roll_168"] = hist["MW_roll_168_base"].reindex(df["roll_time"]).to_numpy()

    df["MW_std_24"] = hist["MW_std_24_base"].reindex(df["roll_time"]).to_numpy()
    df["MW_std_168"] = hist["MW_std_168_base"].reindex(df["roll_time"]).to_numpy()

    feature_cols += [
        "MW_roll_24", "MW_roll_48", "MW_roll_72", "MW_roll_168",
        "MW_std_24", "MW_std_168"
    ]

    # drop rows without enough history
    df = df.dropna(subset=feature_cols).copy()

    # optional: remove helper columns
    df.drop(columns=["day_shift", "roll_time"] + helper_cols, inplace=True)

    return df


zonal_wcl = (
    zonal_wcl
    .groupby("agg_NodeName", group_keys=False)
    .apply(add_zone_features)
    .reset_index(drop=True)
)

In [52]:
zonal_wcl.tail()

,datetime_ending_ept,zone,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,snow_depth,...,MW_lag_96,MW_lag_120,MW_lag_144,MW_lag_168,MW_roll_24,MW_roll_48,MW_roll_72,MW_roll_168,MW_std_24,MW_std_168
568035,2026-05-04 21:00:00,RECO,17.4,14.9,7.1,51.0,0.0,0.0,0.0,0.0,...,154.20599,158.202,143.44200,145.71201,130.911558,134.290459,135.800195,133.168357,10.508353,13.001639
568036,2026-05-04 22:00:00,RECO,16.6,14.4,6.9,53.0,0.0,0.0,0.0,0.0,...,148.85800,151.592,140.33701,143.10699,130.531444,134.158408,135.705147,133.181366,9.653061,13.015352
568037,2026-05-04 23:00:00,RECO,15.9,13.4,6.6,54.0,0.0,0.0,0.0,0.0,...,139.94200,140.547,133.10001,137.23199,130.224373,134.031220,135.638993,133.189485,9.043710,13.022004
568038,2026-05-05 00:00:00,RECO,15.4,12.8,6.4,55.0,0.0,0.0,0.0,0.0,...,129.41000,129.770,124.78800,131.30000,129.983328,134.002303,135.645438,133.209426,8.663865,13.030793
568039,2026-05-05 10:00:00,RECO,20.6,17.7,8.8,47.0,0.0,0.0,0.0,0.0,...,134.68800,136.248,134.75700,119.05800,128.894945,132.373295,135.016159,133.307294,9.501649,12.944627


## LightGBM regression

In [53]:
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

results = []
feat_imp_results = []

start_date = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize() # pd.Timestamp("2026-03-25")
end_date   = start_date + pd.Timedelta(days=2) - pd.Timedelta(hours=1) # pd.Timestamp("2026-03-26")

zones = sorted(zonal_wcl["zone"].dropna().unique())

for zone in zones:
    print(f"\n========== Running zone: {zone} ==========")

    zone_df = (
        zonal_wcl[zonal_wcl["zone"] == zone]
        .sort_values("datetime_ending_ept")
        .copy()
    )

    zone_fcst = (
        zone_load_forecast[zone_load_forecast["zone"] == zone]
        .sort_values("forecast_datetime_ending_ept")
        .copy()
    )

    current_date = start_date

    while current_date <= end_date:

        print(f"Running for zone={zone}, date={current_date.date()}")

        # cutoff = previous day at 10:00
        cutoff = (current_date - pd.Timedelta(days=1)).replace(hour=10)

        # train set: only information available by cutoff
        train_df = zone_df[
            zone_df["datetime_ending_ept"] <= cutoff
        ].copy()

        # trimming: compute thresholds from training data only
        lower = train_df["MW"].quantile(0.01)
        upper = train_df["MW"].quantile(0.99)

        # trim training set
        train_df = train_df[
            (train_df["MW"] >= lower) &
            (train_df["MW"] <= upper)
        ].copy()


        # test set: full target day
        test_start = current_date.replace(hour=1)
        test_end   = current_date.replace(hour=23) + pd.Timedelta(hours=1)

        test_df = zone_df[
            (zone_df["datetime_ending_ept"] >= test_start) &
            (zone_df["datetime_ending_ept"] <= test_end)
        ].copy()

        # skip if empty
        if train_df.empty or test_df.empty:
            print("Skipping because train/test is empty")
            current_date += pd.Timedelta(days=1)
            continue

        target = "MW"

        drop_cols = ["MW", "datetime_ending_ept", "zone", "date"]
        features = [col for col in train_df.columns if col not in drop_cols]

        # time-based validation split from training data
        # use the most recent 14 days of hourly data as validation
        valid_hours = 24 * 14

        if len(train_df) <= valid_hours:
            print("Skipping because not enough training history for validation split")
            current_date += pd.Timedelta(days=1)
            continue

        train_part = train_df.iloc[:-valid_hours].copy()
        valid_part = train_df.iloc[-valid_hours:].copy()

        if train_part.empty or valid_part.empty:
            print("Skipping because train_part/valid_part is empty")
            current_date += pd.Timedelta(days=1)
            continue

        X_train = train_part[features]
        y_train = train_part[target]

        X_valid = valid_part[features]
        y_valid = valid_part[target]

        X_test = test_df[features]

        model = LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.03,
            num_leaves=64,
            max_depth=-1,
            min_child_samples=50,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(100),
                lgb.log_evaluation(100)
            ]
        )

        # feature importance for this day + zone
        feat_imp = pd.DataFrame({
            "zone": zone,
            "date": current_date,
            "feature": X_train.columns,
            "importance": model.feature_importances_
        }).sort_values(by="importance", ascending=False)

        feat_imp["rank"] = range(1, len(feat_imp) + 1)
        feat_imp_results.append(feat_imp)

        test_df = test_df.copy()
        test_df["MW_pred"] = model.predict(X_test)

        # zonal forecast for same target day
        forecast_df = (
            zone_fcst[
                (zone_fcst["forecast_datetime_ending_ept"] >= test_start) &
                (zone_fcst["forecast_datetime_ending_ept"] <= test_end)
            ]
            .sort_values("forecast_datetime_ending_ept")
            .copy()
        )

        forecast_df = forecast_df.rename(columns={
            "forecast_datetime_ending_ept": "datetime_ending_ept"
        })

        forecast_df["datetime_ending_ept"] = pd.to_datetime(
            forecast_df["datetime_ending_ept"]
        )

        compare_df = pd.merge(
            test_df,
            forecast_df[["datetime_ending_ept", "forecast_load_mw"]],
            on="datetime_ending_ept",
            how="inner"
        )

        if compare_df.empty:
            print("Skipping because compare_df is empty after forecast merge")
            current_date += pd.Timedelta(days=1)
            continue

        mae = mean_absolute_error(compare_df["forecast_load_mw"], compare_df["MW_pred"])
        rmse = np.sqrt(mean_squared_error(compare_df["forecast_load_mw"], compare_df["MW_pred"]))

        print(f"MAE: {mae:.2f}, RMSE: {rmse:.2f}")

        compare_df["zone"] = zone
        compare_df["date"] = current_date
        compare_df["MAE"] = mae
        compare_df["RMSE"] = rmse

        results.append(compare_df)

        current_date += pd.Timedelta(days=1)

# combine all days / zones
final_results = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
final_feat_importance = pd.concat(feat_imp_results, ignore_index=True) if feat_imp_results else pd.DataFrame()


========== Running zone: AE ==========
Running for zone=AE, date=2026-05-03
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003652 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13201
[LightGBM] [Info] Number of data points in the train set: 27431, number of used features: 72
[LightGBM] [Info] Start training from score 1132.592695
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 68.5772	valid_0's l2: 4702.83
[200]	valid_0's rmse: 54.1432	valid_0's l2: 2931.49
[300]	valid_0's rmse: 52.4262	valid_0's l2: 2748.5
[400]	valid_0's rmse: 51.6945	valid_0's l2: 2672.32
[500]	valid_0's rmse: 50.7148	valid_0's l2: 2571.99
[600]	valid_0's rmse: 50.4816	valid_0's l2: 2548.39
[700]	valid_0's rmse: 50.2423	valid_0's l2: 2524.29
[800]	valid_0's rmse: 50.1748	valid_0's l2: 2517.51
[900]	valid_0's rmse: 50.1183	valid_0's l2: 2511.84
[1000]	valid_0's rmse: 50.3061	valid_0's l2: 253

In [54]:
zone_summary = []

for zone, df in final_results.groupby("zone"):

    mean_mae = df["MAE"].mean()
    mean_rmse = df["RMSE"].mean()

    zone_summary.append({
        "zone": zone,
        "mean_mae": mean_mae,
        "mean_rmse": mean_rmse,
    })

zone_summary_df = pd.DataFrame(zone_summary)
zone_summary_df

,zone,mean_mae,mean_rmse
0,AE,37.439919,51.844991
1,AEP,165.556396,203.472167
2,APS,124.044094,142.745879
3,ATSI,73.916215,89.590464
4,BGE,139.703402,164.746196
5,COMED,114.250508,145.827075
6,DAYTON,31.867841,37.772444
7,DPL,60.240822,71.741973
8,DQE,13.676949,17.181609
9,DUKE,59.673805,69.355036


In [55]:
import plotly.graph_objects as go

selected_zones = ["VEPCO"]  # choose zones to plot

# selected_zones = ['AE', 'AEP', 'APS', 'ATSI', 'BGE', 'COMED', 'DAYTON', 'DPL', 'DQE', 'DUKE', 'EKPC', 'JCPL', 'METED', 'PECO', 'PENLC', 'PEPCO', 'PL', 'PS', 'RECO', 'VEPCO']

for zone in selected_zones:

    plot_df = (
        final_results[final_results["zone"] == zone]
        .sort_values("datetime_ending_ept")
    )

    if plot_df.empty:
        print(f"Skipping {zone} (no data)")
        continue

    mean_mae = plot_df["MAE"].mean()
    mean_rmse = plot_df["RMSE"].mean()

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=plot_df["datetime_ending_ept"],
        y=plot_df["MW_pred"],
        mode="lines",
        name="Model"
    ))

    fig.add_trace(go.Scatter(
        x=plot_df["datetime_ending_ept"],
        y=plot_df["forecast_load_mw"],
        mode="lines",
        name="PJM"
    ))

    fig.update_layout(
        title=(
            f"{zone} | Model vs PJM<br>"
            f"Model → MAE: {mean_mae:.2f}, RMSE: {mean_rmse:.2f} "
        ),
        xaxis_title="Time",
        yaxis_title="MW",
        template="plotly_white"
    )

    fig.show()

## save

In [56]:
final_results.head()

,datetime_ending_ept,zone,temperature_2m,apparent_temperature,dew_point_2m,relative_humidity_2m,precipitation,rain,snowfall,snow_depth,...,MW_roll_24,MW_roll_48,MW_roll_72,MW_roll_168,MW_std_24,MW_std_168,MW_pred,forecast_load_mw,MAE,RMSE
0,2026-05-03 01:00:00,AE,8.8,4.7,-2.2,46.0,0.0,0.0,0.0,0.0,...,763.337680,818.515863,837.019617,859.701766,195.982557,141.317007,835.061011,853.0,43.500096,54.41477
1,2026-05-03 02:00:00,AE,7.8,3.5,-2.8,47.0,0.0,0.0,0.0,0.0,...,763.603588,818.432709,837.116140,859.806097,196.111084,141.308804,795.741709,821.0,43.500096,54.41477
2,2026-05-03 03:00:00,AE,6.9,3.0,-2.3,52.0,0.0,0.0,0.0,0.0,...,763.644276,818.278794,837.112558,859.885455,196.123718,141.283922,799.522555,800.0,43.500096,54.41477
3,2026-05-03 04:00:00,AE,5.8,2.1,-2.1,57.0,0.0,0.0,0.0,0.0,...,763.516547,818.061174,837.075409,859.968724,196.095018,141.246774,802.181244,791.0,43.500096,54.41477
4,2026-05-03 05:00:00,AE,5.1,1.1,-2.0,60.0,0.0,0.0,0.0,0.0,...,762.948949,817.737243,836.983649,860.044279,195.990242,141.206274,825.617183,795.0,43.500096,54.41477


In [ ]:
# today = datetime.today() - timedelta(days=offset)

# start_str = today.strftime("%m%d")                 # 0325
# end_str = (today + timedelta(days=1)).strftime("%m%d")  # 0326
# run_str = today.strftime("%y%m%d")                 # 260325

# final_results[
#     ['datetime_ending_ept', 'zone', 'date', 'hour', 'MW_pred', 'forecast_load_mw']
# ].to_csv(
#     f'data/prediction/zone_prediction_{start_str}_to_{end_str}_at_{run_str}.csv',
#     index=False
# )